# 1) Setup


### a) Standard imports

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import regex as re
import sys
from pathlib import Path
import os

# Standard imports for data processing and visualization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px
import gensim

print("Imports successful")

Imports successful


### b) Custom modules

In [2]:
# Import custom modules
# Add modules to path
sys.path.insert(0, '/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/modules')

import pca_util
import importlib
import semaxis_util
import semanalysis_util
import helpers
importlib.reload(pca_util)
importlib.reload(semaxis_util)
importlib.reload(semanalysis_util)
importlib.reload(helpers)

from pca_util import create_actor_action_matrix
from semaxis_util import SemAxis, anch2vec, anch2conceptvec, axis_parallelism, pair_parallelism, find_antonym, find_antonyms_fullsearch
from semanalysis_util import (
    actor_embeddings_from_w2v_entities,
    actor_embd, actor_proj, compare_semantic_to_pca,
    association_matrix, compare_verb_loadings, visualize_projection,
)
from helpers import *


print("Imports successful")


Imports successful


### c) Word2Vec model

In [3]:
# Trained Word2Vec (saved under w2v/models/<MODEL_NAME>/ by train_w2v_cpu.py)
MODEL_NAME = "2_w2v_min10_de"

print("Loading trained Word2Vec...")
w2v_model = helpers.load_trained_w2v_keyed_vectors(MODEL_NAME)
# L2-normalize vectors in-place for semantic-axis / projection math
if hasattr(w2v_model, "init_sims"):
    w2v_model.init_sims(replace=True)
    print("Normalized Vectors to unit length")
else:
    print("Cannot normalize: init_sims missing")

print(f"Loaded run {MODEL_NAME!r}. Vocabulary size: {len(w2v_model):,}")
print(f"Vector size: {w2v_model.vector_size}")

# Optional: maps for PCA (canonical) ↔ w2v underscore tokens
token_to_canonical = helpers.load_w2v_token_to_canonical()
canonical_to_w2v = helpers.load_canonical_to_w2v_token()
print(f"Entity map: {len(token_to_canonical)} w2v tokens → canonical")


Loading trained Word2Vec...


/scratch/local/jobs/49216580/ipykernel_337361/1285360398.py:8: DeprecationWarning: Call to deprecated `init_sims` (Use fill_norms() instead. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)


Normalized Vectors to unit length
Loaded run '2_w2v_min10_de'. Vocabulary size: 484,571
Vector size: 300
Entity map: 132 w2v tokens → canonical


Let's inspect the word2vec model for quality.

In [4]:
# Define tokens for easy adjustment
similarity_token = "angela_merkel"
analogy_tokens = {
    "positive": ["lüge", "wahr"],
    "negative": ["falsch"],
}

# Find the 10 closest tokens to the selected token
most_similar_results = w2v_model.most_similar(similarity_token, topn=10)
print(f"Tokens closest to '{similarity_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

print("\nWord analogies:")
try:
    analogy_results = w2v_model.most_similar(
        positive=analogy_tokens["positive"],
        negative=analogy_tokens["negative"],
        topn=10
    )
    positive_str = " + ".join(analogy_tokens["positive"])
    negative_str = " - " + " - ".join(analogy_tokens["negative"]) if analogy_tokens["negative"] else ""
    print(f"{positive_str}{negative_str} is closest to:")
    for tok, sim_score in analogy_results:
        print(f"{tok:20s} {sim_score:.3f}")
except KeyError as e:
    print(f"Token not in vocabulary: {e}")

Tokens closest to 'angela_merkel':
bundeskanzlerin      0.825
kanzlerin            0.819
merkels              0.733
angela               0.601
jens_spahn           0.600
mehrkill             0.599
günstlername         0.587
merkill              0.584
kazcmierzak          0.571
dosvedanja           0.569

Word analogies:
lüge + wahr - falsch is closest to:
wahrheit             0.468
pulverine            0.445
lügen                0.417
verschwörungstheorie 0.414
wetterdarstellung    0.406
entvölk              0.389
erdölknappheit       0.387
verschwöhrungstheroie 0.385
bääääääääähm         0.383
cheamtrails          0.383


Note that the German model has a vocabulary (484K) that's about 2.7x as large as in English (178K). 

That difference is likely due to a mixture of German peculiarities, including more unique compound words (e.g., Impfplicht), noun inflections (Nominative, Genitive, etc.), and verb conjugations. Stemming might help here, but that'd require re-running the pipeline in both languages with significantly increased computational overhead (from using SpaCy). Additionally, none of the extant literature on word embeddings for cultural analysis recommends stemming; Boutyline and Arseniev-Kohler (2025) in fact actively discourage it, instead suggesting the construction of axes using morphological variants for added robustness.

I will therefore stick to the current approach. I'd re-process the text only if the current vocabulary presents significant issues during analysis.

In [5]:
# Check vocabulary composition
import collections

# Sample vocabulary to see what's inflating German count
german_vocab_sample = list(w2v_model.index_to_key[:100])
print(german_vocab_sample)

# Check for compound patterns
compounds = [w for w in w2v_model.index_to_key if len(w) > 15]
print(f"Long words (likely compounds): {len(compounds)}")
print(compounds[:50])

# Compare frequency distributions
eb_freqs = [w2v_model.get_vecattr(w, "count") for w in w2v_model.index_to_key[:1000]]

['die', 'und', 'der', 'in', 'das', 'ist', 'zu', 'es', 'von', 'den', 'nicht', 'mit', 'sie', 'auf', 'für', 'ich', 'ein', 'sich', 'eine', 'dass', 'wir', 'im', 'auch', 'wird', 'dem', 'werden', 'sind', 'an', 'des', 'hat', 'wie', 'als', 'haben', 'was', 'um', 'aus', 'so', 'er', 'noch', 'mehr', 'nur', 'oder', 'wenn', 'bei', 'man', 'aber', 'über', 'uns', 'nach', 'am', 'vor', 'menschen', 'hier', 'diese', 'einen', 'einer', 'alle', 'ihr', 'zum', 'kann', 'du', 'jetzt', 'sein', 'wurde', 'einem', 'dann', 'war', 'da', 'gegen', 'immer', 'durch', 'zur', 'keine', 'alles', 'schon', 'gibt', 'können', 'wieder', 'dieser', 'mal', 'bis', 'ihre', 'corona', 'habe', 'euch', 'geht', 'ja', 'selbst', 'vom', 'sehr', 'mir', 'die_welt', 'doch', 'deutschland', 'unter', 'viele', 'unsere', 'wurden', 'mich', 'kanal']
Long words (likely compounds): 79013
['world_health_organization', 'world_economic_forum', 'federal_bureau_of_investigation', 'robert_koch_institut', 'central_intelligence_agency', 'rothschild_family', 'center

# 2) Construct Semantic Axes

In this section, I will:
1. Define antonym pairs representing semantic dimensions
2. Create semantic axes using the `SemAxis` class
3. Evaluate axis quality using parallelism metrics
4. Refine axes by removing weak pairs or adding strong ones

## Epistemic Action Axis

In [105]:
# Summary of the epistemic-action axis
# Create semantic axis using the SemAxis class defined in semaxis_util.py
epistemic_action_axis = SemAxis(
    [
    ("aufklären", "verschleiern"),
    ('enthüllen', 'verbergen'),
    ("aufdecken", "vertuschen")
], 
    w2v_model, 
    name="reveal_hide_de"
)

# View axis summary
print(epistemic_action_axis.summary())

Semantic Axis: reveal_hide_de
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.251

Best pairs (highest parallelism):
  ('aufdecken', 'vertuschen'): 0.292
  ('aufklären', 'verschleiern'): 0.236
  ('enthüllen', 'verbergen'): 0.224

Worst pairs (lowest parallelism):
  ('enthüllen', 'verbergen'): 0.224
  ('aufklären', 'verschleiern'): 0.236
  ('aufdecken', 'vertuschen'): 0.292


In [67]:
# Looking for top N best antonyms
f, r = epistemic_action_axis.find_antonyms_fullsearch("beleuchten", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

# 'kaschieren' suits quite well as a translation of 'obfuscate'

Word-Score: kaschieren - 0.32482833523608423
Word-Score: verdecken - 0.32194494261891793
Word-Score: ausrede - 0.2853927856815429
Word-Score: umzudeuten - 0.2828614204045268
Word-Score: schnullis - 0.28195206856751603
Word-Score: chutzmasken - 0.28185877258816766
Word-Score: verheimlichen - 0.2634101723070589
Word-Score: abzulenken - 0.26223836688397834
Word-Score: überdecken - 0.26187514648127885
Word-Score: schurkereien - 0.26107691294760954
Word-Score: hinauszuzögern - 0.259146168379368
Word-Score: impfstoffdefinition - 0.2590513225429791
Word-Score: pandemieerzählung - 0.2582220131743328
Word-Score: tatsächlichkeiten - 0.2580375348245127
Word-Score: waschen - 0.257353773963576
Word-Score: kreditsystems - 0.25468748242265465
Word-Score: schlichtweg - 0.2543035504057074
Word-Score: sensationsmedien - 0.252406377927922
Word-Score: admiralitätsgesetze - 0.2512813299987611
Word-Score: anzuwerfen - 0.25024712065450466
Word-Score: fledermauserreger - 0.2497237430265727
Word-Score: relativ

In [106]:
# Add pair
epistemic_action_axis = epistemic_action_axis.add_pair(('beleuchten', 'kaschieren'))
print(epistemic_action_axis.summary())

Semantic Axis: reveal_hide_de
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.240

Best pairs (highest parallelism):
  ('aufdecken', 'vertuschen'): 0.284
  ('aufklären', 'verschleiern'): 0.239
  ('beleuchten', 'kaschieren'): 0.229

Worst pairs (lowest parallelism):
  ('enthüllen', 'verbergen'): 0.207
  ('beleuchten', 'kaschieren'): 0.229
  ('aufklären', 'verschleiern'): 0.239


In [110]:
# Looking for top N best antonyms
f, r = epistemic_action_axis.find_antonyms_fullsearch("erleuchten", top_n=100, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

# leaken-verdecken gets quite close to leak-suppress

Word-Score: weltenschicksal - 0.3442709917190693
Word-Score: cometa - 0.33916693132857545
Word-Score: vademecum - 0.33831404499700685
Word-Score: iverse - 0.3364919279405376
Word-Score: decoded - 0.33615346785497197
Word-Score: 2569b - 0.3344346266896061
Word-Score: elektronikbereich - 0.3326156969796873
Word-Score: wahrheitskampf - 0.3317192819735382
Word-Score: selbstfindungsreise - 0.3316321644734147
Word-Score: ankündige - 0.33138391550178437
Word-Score: gurney - 0.3313401765984284
Word-Score: mahoney - 0.3300560468529395
Word-Score: brittanys - 0.3268268391577722
Word-Score: zeitkritischen - 0.32659823889865425
Word-Score: medienkritische - 0.32615039764014375
Word-Score: krisenratinfo - 0.3242664832397757
Word-Score: sendend - 0.32375831870513994
Word-Score: sommerlektüre - 0.32348791952971495
Word-Score: impfreport - 0.3226191686969661
Word-Score: coronadebatte - 0.32248413065440346
Word-Score: wissensfragmente - 0.32246141373053994
Word-Score: phantastischen - 0.322460476700864

In [109]:
# Find new pair to add
anchor = 'erleuchten'
candidates = ['täuschen', 'verheimlichen', 'verdrehen', 'verdecken', 'leugnen', 'ablenken', 'relativieren']

best = epistemic_action_axis.find_best_antonym(anchor, candidates)
print(best)
print(f"Best new pair: {anchor}-{best[0][0]} ({best[0][1]:.3f})")

[('verdecken', 0.21776938140392305), ('verheimlichen', 0.20316233709454537), ('relativieren', 0.19763420298695564), ('täuschen', 0.19563211761415006), ('leugnen', 0.193355218321085)]
Best new pair: erleuchten-verdecken (0.218)


In [82]:
# Add pair
epistemic_action_axis = epistemic_action_axis.add_pair(('offenlegen', 'verdecken'))
print(epistemic_action_axis.summary())

Semantic Axis: reveal_hide
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.240

Best pairs (highest parallelism):
  ('aufdecken', 'vertuschen'): 0.284
  ('aufklären', 'verschleiern'): 0.239
  ('beleuchten', 'kaschieren'): 0.229

Worst pairs (lowest parallelism):
  ('enthüllen', 'verbergen'): 0.207
  ('beleuchten', 'kaschieren'): 0.229
  ('aufklären', 'verschleiern'): 0.239


## Hidden: Epistemic-action conjugations

Below is the code used to experiment with adding/removing individual combinations from the conjugations. While each one may individually fit the epistemic axis fairly well, their interaction in the axis can lead to low overall parallelism scores. Ultimately, I kept only the `suppresses-leaks` pair; this ensured that all pairs in the axis kept an individual score above 0.2 (strong).

In [ ]:
# Checking conjugations
conj = [('suppressing', 'leaking'), ('suppresses', 'leaks'), ('concealing', 'exposing'), ('conceals', 'exposes'), ('hiding', 'revealing'), ('hides', 'reveals')]
for bad, good in conj:
    candidates = [bad]
    best = epistemic_action_axis.find_best_antonym(good, candidates)
    print(f"{good} - {best[0][0]} ({best[0][1]:.3f})")

leaking - suppressing (0.243)
leaks - suppresses (0.242)
exposing - concealing (0.208)
exposes - conceals (0.171)
revealing - hiding (0.157)
reveals - hides (0.198)


In [49]:
# Adding one pair at a time to check if interaction between pairs is significant
epistemic_action_axis = epistemic_action_axis.add_pair(('hides', 'reveals'))

# Calculate overall parallelism
summary = epistemic_action_axis.summary()
match = re.search(r"Overall parallelism:\s*([-\d\.]+)", summary)
parallelism_score = float(match.group(1))
print(f"Overall parallelism: {parallelism_score}")

# Rechecking conjugations
conj = [('hiding', 'revealing')]
for bad, good in conj:
    candidates = [bad]
    best = epistemic_action_axis.find_best_antonym(good, candidates)
    print(f"{good} - {best[0][0]} ({best[0][1]:.3f})")

# Removing weaker pairs
# epistemic_action_axis = epistemic_action_axis.remove_pair(('conceals', 'exposes'))

Overall parallelism: 0.205
revealing - hiding (0.173)


In [55]:
epistemic_action_axis = epistemic_action_axis.remove_pair(('hides', 'reveals'))
# epistemic_action_axis = epistemic_action_axis.add_pair(('conceals', 'exposes'))
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.242

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.315
  ('suppresses', 'leaks'): 0.244
  ('conceal', 'expose'): 0.207

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.203
  ('conceal', 'expose'): 0.207
  ('suppresses', 'leaks'): 0.244


In [ ]:

for pair in conj:
    if pair not in [('exposes', 'conceals'), ('revealing', 'hiding')]:
        epistemic_action_axis = epistemic_action_axis.add_pair(pair)
        print(epistemic_action_axis.summary())
        i += 1
        if i > 1:
            break
print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.241

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.249
  ('conceal', 'expose'): 0.248
  ('hide', 'reveal'): 0.226

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.226
  ('conceal', 'expose'): 0.248
  ('suppress', 'leak'): 0.249


In [ ]:
# Seems like there is some interaction between the pairs, so I'll remove the weaker ones.
for pair in conj:
    if pair not in [('exposes', 'conceals'), ('revealing', 'hiding')]:
        epistemic_action_axis = epistemic_action_axis.remove_pair(pair)

print(epistemic_action_axis.summary())

Semantic Axis: epistemic_action
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.242

Best pairs (highest parallelism):
  ('suppress', 'leak'): 0.315
  ('suppresses', 'leaks'): 0.244
  ('conceal', 'expose'): 0.207

Worst pairs (lowest parallelism):
  ('hide', 'reveal'): 0.203
  ('conceal', 'expose'): 0.207
  ('suppresses', 'leaks'): 0.244


Note that the SemAxis class also allows us to more directly examine the parallelism of antonym pairs.

In [6]:
# Check which pairs fit well and which don't
print("Best pairs (highest parallelism):")
for pair, score in epistemic_action_axis.get_best_pairs(3):
    print(f"  {pair[0]:15s} - {pair[1]:15s} : {score:.3f}")

print("\nWorst pairs (lowest parallelism):")
for pair, score in epistemic_action_axis.get_worst_pairs(3):
    print(f"  {pair[0]:15s} - {pair[1]:15s} : {score:.3f}")

Best pairs (highest parallelism):
  hide            - reveal          : 0.230
  conceal         - expose          : 0.208
  suppress        - leak            : 0.207

Worst pairs (lowest parallelism):
  obfuscate       - illuminate      : 0.171
  suppress        - leak            : 0.207
  conceal         - expose          : 0.208


## Epistemic-action Axis Overview

In [112]:
epistemic_action_axis = SemAxis(
    [
    ("aufklären", "verschleiern"),
    ('enthüllen', 'verbergen'),
    ("aufdecken", "vertuschen"),
    ('beleuchten', 'kaschieren')
], 
    w2v_model, 
    name="reveal_hide_de"
)

# View axis summary
print(epistemic_action_axis.summary())

Semantic Axis: reveal_hide_de
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.240

Best pairs (highest parallelism):
  ('aufdecken', 'vertuschen'): 0.284
  ('aufklären', 'verschleiern'): 0.239
  ('beleuchten', 'kaschieren'): 0.229

Worst pairs (lowest parallelism):
  ('enthüllen', 'verbergen'): 0.207
  ('beleuchten', 'kaschieren'): 0.229
  ('aufklären', 'verschleiern'): 0.239


## True-False Axis

In [123]:
# Construct new true-false axis; start big then whittle down
true_false_axis = SemAxis(
    [("wahr", "falsch")
    ], 
    w2v_model, 
    name="true_false_de"
)
print(true_false_axis.summary())

Semantic Axis: true_false_de
Number of antonym pairs: 1
Concept vector dimension: 300


In [140]:
f, r = true_false_axis.find_antonyms_fullsearch("irreführend", top_n=100, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: flugkapitan - 0.43885223991632266
Word-Score: pulverine - 0.426948527091082
Word-Score: verschwörungstheorie - 0.4246068667608656
Word-Score: frei - 0.4146553157074475
Word-Score: denk - 0.413229740477377
Word-Score: freiheit - 0.41092947805796787
Word-Score: rankten - 0.40986099595917835
Word-Score: maskenbullshit - 0.4011679276717677
Word-Score: tobag - 0.3998816675753564
Word-Score: verschwöööhrungstheorie - 0.3991482779724942
Word-Score: achende - 0.3987871273452746
Word-Score: remonstrationsrechts - 0.3981868296020151
Word-Score: warfighters - 0.3964462570253169
Word-Score: cheamtrails - 0.395600523018801
Word-Score: möbelstücks - 0.3935241868235504
Word-Score: vollmonden - 0.3910706486687538
Word-Score: hr_c - 0.39078506129577106
Word-Score: spardabank - 0.39077423734747174
Word-Score: verschwörungstheorien - 0.39028796283371514
Word-Score: annerkanntes - 0.39011551739326467
Word-Score: umweltkon - 0.3898991370076206
Word-Score: impfsüchtigen - 0.3887384061415049
Word

In [142]:
# Add pair
true_false_axis = true_false_axis.add_pair(('wahrhaftig', 'irreführend'))
print(true_false_axis.summary())

Semantic Axis: true_false_de
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.279

Best pairs (highest parallelism):
  ('wahr', 'falsch'): 0.279
  ('wahrhaftig', 'irreführend'): 0.279

Worst pairs (lowest parallelism):
  ('wahr', 'falsch'): 0.279
  ('wahrhaftig', 'irreführend'): 0.279


In [ ]:
f, r = true_false_axis.find_antonyms_fullsearch("authentisch", top_n=100, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

In [ ]:
# Add pair
true_false_axis = true_false_axis.add_pair(('authentisch', 'betrügerisch'))
print(true_false_axis.summary())

Semantic Axis: true_false_de
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.251

Best pairs (highest parallelism):
  ('wahr', 'falsch'): 0.258
  ('wahrhaftig', 'irreführend'): 0.258
  ('authentisch', 'betrügerisch'): 0.238

Worst pairs (lowest parallelism):
  ('authentisch', 'betrügerisch'): 0.238
  ('wahrhaftig', 'irreführend'): 0.258
  ('wahr', 'falsch'): 0.258


In [176]:
# Define tokens for easy adjustment
similarity_token = "glaubwürdig"

# Find the N closest tokens to the selected token
most_similar_results = w2v_model.most_similar(similarity_token, topn=20)
print(f"Tokens closest to '{similarity_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'glaubwürdig':
unglaubwürdig        0.541
vertrauenswürdig     0.519
wertevernichter      0.487
seriös               0.486
überzeugend          0.478
unseriös             0.464
fragwürdig           0.460
behauptungen         0.452
authentisch          0.448
schwurblerverse      0.447
unsympathisch        0.439
merkwürdig           0.436
kompetent            0.432
biowaffentheorie     0.429
klimawarner          0.426
koscha               0.424
sympathisch          0.423
affronts             0.423
zweifelhaft          0.420
unwahr               0.420


In [187]:
f, r = true_false_axis.find_antonyms_fullsearch("glaubwürdig", top_n=100, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: irreführende - 0.33058565516822214
Word-Score: uneinheitliche - 0.31848864025331214
Word-Score: fehlerhafte - 0.31636465368510047
Word-Score: betrügerische - 0.31138344634362947
Word-Score: fehlerhaften - 0.3110885691945465
Word-Score: fehlerhaft - 0.3098886494379827
Word-Score: unzuverlässig - 0.30736174769552727
Word-Score: vierundsechzigseitigen - 0.29642113502400436
Word-Score: nachgetestet - 0.29209111157923295
Word-Score: unsinnig - 0.29169079685667204
Word-Score: grob - 0.2913350810257037
Word-Score: ausgeteilte - 0.28967582956993815
Word-Score: falsche - 0.28720945360201616
Word-Score: laienpresse - 0.2863734120950254
Word-Score: geeichte - 0.2840736958661483
Word-Score: ungenau - 0.28403212311056414
Word-Score: unnötig - 0.2836227899200254
Word-Score: gehandhabten - 0.2829343253542438
Word-Score: diskutablen - 0.2819308722651918
Word-Score: unrechtmäßinstagram - 0.28130277941970067
Word-Score: durchläufen - 0.2806650642741899
Word-Score: inzidenzberechnung - 0.2803

In [190]:
true_false_axis = true_false_axis.add_pair(('glaubwürdig', 'fehlerhaft'))
print(true_false_axis.summary())

Semantic Axis: true_false_de
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.237

Best pairs (highest parallelism):
  ('wahr', 'falsch'): 0.259
  ('authentisch', 'betrügerisch'): 0.238
  ('wahrhaftig', 'irreführend'): 0.227

Worst pairs (lowest parallelism):
  ('glaubwürdig', 'fehlerhaft'): 0.222
  ('wahrhaftig', 'irreführend'): 0.227
  ('authentisch', 'betrügerisch'): 0.238


In [207]:
# Define tokens for easy adjustment
similarity_token = "wahrheitsgetreu"

# Find the N closest tokens to the selected token
most_similar_results = w2v_model.most_similar(similarity_token, topn=20)
print(f"Tokens closest to '{similarity_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'wahrheitsgetreu':
sachlich             0.412
wahrheitsgrad        0.402
akkurat              0.395
propagandafrei       0.395
ausgewogen           0.381
dauerwerbesendungen  0.379
beigelegtes          0.378
erkrankungsgeschehen 0.378
faktenbasierend      0.377
neutral              0.376
hintergrundrecherchen 0.374
schwurbelein         0.371
unparteiisch         0.370
lassenden            0.368
untergriffigkeiten   0.368
mittmannsgrubers     0.368
umfassend            0.365
journalismus         0.365
fortbildendes        0.364
informieren          0.362


In [208]:
f, r = true_false_axis.find_antonyms_fullsearch("wahrheitsgetreu", top_n=100, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: fehlerhafte - 0.30098856656007456
Word-Score: uneinheitliche - 0.29320200789782513
Word-Score: fehlerhaften - 0.28502646808640053
Word-Score: durchgetesteter - 0.27793522098600365
Word-Score: nachgetestet - 0.2776579481045852
Word-Score: bestätigungstest - 0.2741357031151389
Word-Score: irreführende - 0.2732603229880457
Word-Score: pcr - 0.27283306358260184
Word-Score: grob - 0.27140451248290587
Word-Score: ellume - 0.2696631989881577
Word-Score: zyklusschwellenwert - 0.2689138235109785
Word-Score: geeichte - 0.268510779670412
Word-Score: doppelstimmzettel - 0.2672847750385383
Word-Score: unzuverlässig - 0.26467657365814223
Word-Score: diskutablen - 0.26408512284318675
Word-Score: unnötig - 0.26392685613632955
Word-Score: betrügerische - 0.2634999593426598
Word-Score: betrügerischen - 0.2633415761572214
Word-Score: tests - 0.26284578422210136
Word-Score: hochmanipuluert - 0.2624257667522524
Word-Score: ungenau - 0.2622245573614534
Word-Score: konstruktionsprinzip - 0.261770

In [ ]:
# Seems good enough
print(true_false_axis.add_pair(('wahrheitsgetreu', 'ungenau')).summary())

Semantic Axis: true_false_de
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.210

Best pairs (highest parallelism):
  ('wahr', 'falsch'): 0.234
  ('wahrhaftig', 'irreführend'): 0.222
  ('authentisch', 'betrügerisch'): 0.215

Worst pairs (lowest parallelism):
  ('wahrheitsgetreu', 'ungenau'): 0.170
  ('glaubwürdig', 'fehlerhaft'): 0.209
  ('authentisch', 'betrügerisch'): 0.215


## True-False Axis Overview

In [56]:
true_false_axis = SemAxis(
    [("wahr", "falsch"),
    ("wahrhaftig", "irreführend"),
    ("authentisch", "betrügerisch"),
    ("wahrheitsgetreu", "ungenau"),
    ("glaubwürdig", "fehlerhaft")], 
    w2v_model, 
    name="true_false_de"
)
print(true_false_axis.summary())

Semantic Axis: true_false_de
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.210

Best pairs (highest parallelism):
  ('wahr', 'falsch'): 0.234
  ('wahrhaftig', 'irreführend'): 0.222
  ('authentisch', 'betrügerisch'): 0.215

Worst pairs (lowest parallelism):
  ('wahrheitsgetreu', 'ungenau'): 0.170
  ('glaubwürdig', 'fehlerhaft'): 0.209
  ('authentisch', 'betrügerisch'): 0.215


## Good-Evil Axis

Now exploring a good-evil axis. Note that I am using good-evil rather than good-bad to operationalize an explicitly moral evaluation.

In [41]:
good_evil_axis = SemAxis(
    [
        ("gut", "böse")
    ], 
    w2v_model, 
    name="good_evil_de"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil_de
Number of antonym pairs: 1
Concept vector dimension: 300


In [29]:
# Define tokens for easy adjustment
similarity_token = "böse"

# Find the N closest tokens to the selected token
most_similar_results = w2v_model.most_similar(similarity_token, topn=40)
print(f"Tokens closest to '{similarity_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'böse':
q4481                0.591
bösen                0.586
boese                0.534
zibzackmini          0.512
q4461                0.506
2223b                0.494
abgrundtief          0.492
azarbajan            0.489
satan                0.484
schiach              0.477
2215b                0.474
äber                 0.471
übelbösen            0.470
orangenmann          0.470
nocturnal            0.469
ezlrss1woaaz7ls      0.468
satanische           0.468
lotr                 0.464
verwirrer            0.460
gehörnt              0.459
abgrundtiefste       0.457
krankheitsübertrager 0.456
hennemann            0.455
2471b                0.452
böser                0.451
schöpferlose         0.449
_offenbaren_         0.449
medienschreiberling  0.446
iiiiemlich           0.446
purste               0.446
einkaufsfallen       0.442
sturmfrei            0.441
wahlpanik            0.441
entquellen           0.441
unenthaltsam         0.440
giftverweigerer      0.439
q4

In [25]:
# Looking for top N best antonyms
f, r = good_evil_axis.find_antonyms_fullsearch("niederträchtig", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: besser - 0.3097212016582489
Word-Score: bestens - 0.3004204332828522
Word-Score: prima - 0.29573455452919006
Word-Score: wunderbar - 0.2881109416484833
Word-Score: toll - 0.28245052695274353
Word-Score: schlecht - 0.2792087495326996
Word-Score: hochinteressanten - 0.27069950103759766
Word-Score: entspannt - 0.2682949900627136
Word-Score: hervorragend - 0.2652462422847748
Word-Score: einplant - 0.26266658306121826
Word-Score: zufrieden - 0.26224035024642944
Word-Score: genesenenquote - 0.2616393566131592
Word-Score: mopsen - 0.26151663064956665
Word-Score: feuerwehrdienst - 0.2612597942352295
Word-Score: existensnöten - 0.2601780295372009
Word-Score: supergut - 0.25940245389938354
Word-Score: stb - 0.256946861743927
Word-Score: immunstystem - 0.2566101551055908
Word-Score: rachenpflege - 0.2563326060771942
Word-Score: honoriert - 0.2543677091598511
Word-Score: lösungsstrategien - 0.2540661692619324
Word-Score: schön - 0.25252822041511536
Word-Score: unaufgeregt - 0.252014040

In [42]:
# Add new pair
good_evil_axis = good_evil_axis.add_pair(("honorabel", "niederträchtig"))
print(good_evil_axis.summary())

Semantic Axis: good_evil_de
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.126

Best pairs (highest parallelism):
  ('gut', 'böse'): 0.126
  ('honorabel', 'niederträchtig'): 0.126

Worst pairs (lowest parallelism):
  ('gut', 'böse'): 0.126
  ('honorabel', 'niederträchtig'): 0.126


In [43]:
good_evil_axis = good_evil_axis.add_pair(('aufrichtig', 'pervers'))
print(good_evil_axis.summary())

Semantic Axis: good_evil_de
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.158

Best pairs (highest parallelism):
  ('honorabel', 'niederträchtig'): 0.184
  ('aufrichtig', 'pervers'): 0.173
  ('gut', 'böse'): 0.115

Worst pairs (lowest parallelism):
  ('gut', 'böse'): 0.115
  ('aufrichtig', 'pervers'): 0.173
  ('honorabel', 'niederträchtig'): 0.184


In [44]:
# Looking for top N best antonyms
f, r = good_evil_axis.find_antonyms_fullsearch("tugendhaft", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: holdselig - 0.1664308731754621
Word-Score: honett - 0.16502680877844492
Word-Score: liebreizend - 0.1635183754066626
Word-Score: davia - 0.16102437054117522
Word-Score: ochir - 0.15675999720891318
Word-Score: lilge - 0.15224560350179672
Word-Score: lichste - 0.1520902526875337
Word-Score: rashaev - 0.15171898901462555
Word-Score: ehrenhaft - 0.1513933422975242
Word-Score: zaindi - 0.15032434773941836
Word-Score: spetsnatz - 0.15012219299872717
Word-Score: alkhanov - 0.1500744012494882
Word-Score: oyun - 0.14964556073149046
Word-Score: samed - 0.14963923456768194
Word-Score: bernius - 0.1496335876484712
Word-Score: regressiert - 0.14959794469177723
Word-Score: aronovich - 0.14957869797945023
Word-Score: zarter - 0.14947970832387605
Word-Score: nmd - 0.14932136858503023
Word-Score: umreicht - 0.14928469123939672
Word-Score: treu - 0.14853269482652345
Word-Score: freundlich - 0.14838200186689696
Word-Score: übr - 0.1482877874126037
Word-Score: schätzen - 0.14810875927408537
Wo

In [45]:
good_evil_axis = good_evil_axis.add_pair(('tugendhaft', 'abartig'))
print(good_evil_axis.summary())

Semantic Axis: good_evil_de
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.154

Best pairs (highest parallelism):
  ('honorabel', 'niederträchtig'): 0.190
  ('aufrichtig', 'pervers'): 0.182
  ('tugendhaft', 'abartig'): 0.151

Worst pairs (lowest parallelism):
  ('gut', 'böse'): 0.095
  ('tugendhaft', 'abartig'): 0.151
  ('aufrichtig', 'pervers'): 0.182


## Good-Evil Axis Overview

In [46]:
good_evil_axis = SemAxis(
    [
        ("gut", "böse"),
        ("aufrichtig", "pervers"),
        ("honorabel", "niederträchtig"),
        ("tugendhaft", "abartig")
    ], 
    w2v_model, 
    name="good_evil_de"
)
print(good_evil_axis.summary())

Semantic Axis: good_evil_de
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.154

Best pairs (highest parallelism):
  ('honorabel', 'niederträchtig'): 0.190
  ('aufrichtig', 'pervers'): 0.182
  ('tugendhaft', 'abartig'): 0.151

Worst pairs (lowest parallelism):
  ('gut', 'böse'): 0.095
  ('tugendhaft', 'abartig'): 0.151
  ('aufrichtig', 'pervers'): 0.182


## Holy-Unholy Axis

In [138]:
# An issue with heilig-unheilig is that in German, these words tend to be used for character descriptions,
# not for moral/religious evaluation as in English. For that reason, I'm building the axis with christlich-satanisch, which seems to be 
# used more akin to the Enlish holy-unholy pair in its religious sense.
holy_unholy_axis = SemAxis(
    [
        ("göttlich", "satanisch")
    ], 
    w2v_model, 
    name="holy_unholy_de"
)
print(holy_unholy_axis.summary())

Semantic Axis: holy_unholy_de
Number of antonym pairs: 1
Concept vector dimension: 300


In [139]:
# Looking for top N best antonyms
f, r = holy_unholy_axis.find_antonyms_fullsearch("cherub", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: rituellen - 0.3418786823749542
Word-Score: satanischen - 0.3345222473144531
Word-Score: kaiphas - 0.3279585540294647
Word-Score: ritueller - 0.3236091434955597
Word-Score: athy - 0.31506410241127014
Word-Score: menschenopferung - 0.30623477697372437
Word-Score: satanismus - 0.30224859714508057
Word-Score: tanisch - 0.30029043555259705
Word-Score: pädo - 0.2980133295059204
Word-Score: missbrauch - 0.2964617609977722
Word-Score: kindesmissbräuche - 0.29301872849464417
Word-Score: kindesmisbrauch - 0.2916085124015808
Word-Score: ritue - 0.2906251847743988
Word-Score: kindopfer - 0.28626424074172974
Word-Score: pädophile - 0.28532958030700684
Word-Score: satanischem - 0.2831118404865265
Word-Score: rituel - 0.2819712162017822
Word-Score: ssbrau - 0.28132349252700806
Word-Score: pädokriminalität - 0.2804791331291199
Word-Score: zirkel³ - 0.2789430618286133
Word-Score: pädopartei - 0.27799421548843384
Word-Score: satanischer - 0.2731493413448334
Word-Score: erdächtig - 0.27284109

In [141]:
# Checking parallelism of extended antonym pairs
candidates = [
    ("cherub", "luzifer"),
    ("cherub", "satan"),
    ("cherub", "dämon"),
    ("cherub", "teufel")
]

for spiritual, mundane in candidates:
    try:
        newax = holy_unholy_axis.add_pair((spiritual, mundane))
        print(f"Parallelism with pair {spiritual}-{mundane}: ({newax.parallelism_score:.3f})")
    except ValueError as err:
        print(err)

Parallelism with pair cherub-luzifer: (0.125)
Parallelism with pair cherub-satan: (0.120)
Parallelism with pair cherub-dämon: (0.045)
Parallelism with pair cherub-teufel: (0.194)


In [142]:
holy_unholy_axis = holy_unholy_axis.add_pair(('cherub', 'teufel'))
print(holy_unholy_axis.summary())

Semantic Axis: holy_unholy_de
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.194

Best pairs (highest parallelism):
  ('göttlich', 'satanisch'): 0.194
  ('cherub', 'teufel'): 0.194

Worst pairs (lowest parallelism):
  ('göttlich', 'satanisch'): 0.194
  ('cherub', 'teufel'): 0.194


In [137]:
# Find the 20 nearest neighbors to a word in the w2v_model; gotteslästerlich akin to profane
word = 'himmel'
nearest_neighbors = w2v_model.most_similar(word, topn=50)
print(f"20 nearest neighbors to '{word}':")
for word, similarity in nearest_neighbors:
    print(f"{word}: {similarity:.4f}")

20 nearest neighbors to 'himmel':
wolken: 0.6168
laserprotest: 0.5950
_show: 0.5444
chemflieger: 0.5397
giftstreifen: 0.5353
sternenklarer: 0.5352
princiotta: 0.5338
nachthimmel: 0.5337
wolkenfreier: 0.5330
milchiger: 0.5322
zusprühen: 0.5296
azurblauer: 0.5271
wolkenloser: 0.5267
jedda: 0.5257
maschinenkrieg: 0.5243
armeestil: 0.5240
wolkenlosen: 0.5218
ijob: 0.5201
streifenwolken: 0.5191
wetterpate: 0.5169
himmels: 0.5161
himnel: 0.5152
weisslich: 0.5145
wetterpatent: 0.5121
wolkenlosem: 0.5097
hinmel: 0.5088
tiefstehend: 0.5084
непонятное: 0.5070
wölbst: 0.5062
gleisend: 0.5057
laserkomplex: 0.5038
tieffliegend: 0.5029
streifenfreien: 0.5026
streifenfrei: 0.5015
vertilgte: 0.5002
herbeizubeschwören: 0.4985
freemantv: 0.4983
zugesprühten: 0.4977
свечение: 0.4977
himmeln: 0.4976
172k: 0.4970
abendhimmel: 0.4936
rauchdampf: 0.4919
notfallereignissen: 0.4915
schachbrettartig: 0.4899
erpatente: 0.4899
sternschnuppen: 0.4884
ţhę: 0.4874
ìņţø: 0.4862
babymus: 0.4845


In [145]:
# Looking for top N best antonyms
f, r = holy_unholy_axis.find_antonyms_fullsearch("liebe", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: therapiert¹ - 0.225190540154775
Word-Score: bern² - 0.2136206328868866
Word-Score: zirkel³ - 0.21011974662542343
Word-Score: kindesmisbrauch - 0.20994517455498377
Word-Score: elite⁴ - 0.20811206102371216
Word-Score: verbrecherregierungen - 0.20806998511155447
Word-Score: therapiezimmer - 0.20622423787911734
Word-Score: verbildungssystem - 0.20494548976421356
Word-Score: vollverblödete - 0.20368292927742004
Word-Score: strie - 0.20349760353565216
Word-Score: satanischen - 0.2034370837112268
Word-Score: schamverletzende - 0.20319378872712454
Word-Score: chicen - 0.20143587390581766
Word-Score: satanistischen - 0.20110456148783365
Word-Score: abartig - 0.20059143006801605
Word-Score: satanische - 0.20044265439112982
Word-Score: teufelshochhaus - 0.1998537927865982
Word-Score: täter¹ - 0.19973776241143545
Word-Score: gewalt² - 0.1995809574921926
Word-Score: kinderjagden - 0.1989825243751208
Word-Score: adrenocrome - 0.19859504203001657
Word-Score: tanismus - 0.19846622397502264

In [213]:
# Checking parallelism of extended antonym pairs
candidates = [
    ("geheiligt", "abtrünnig"),
    ("schöpfer", "antichrist"),
    ("eden", "sodom"),
    ("gottesfürchtig", "säkular")
]

for spiritual, mundane in candidates:
    try:
        newax = holy_unholy_axis.add_pair((spiritual, mundane))
        print(f"Parallelism with pair {spiritual}-{mundane}: ({newax.parallelism_score:.3f})")
    except ValueError as err:
        print(err)
   

Parallelism with pair geheiligt-abtrünnig: (0.115)
Parallelism with pair schöpfer-antichrist: (0.168)
Parallelism with pair eden-sodom: (0.111)
Parallelism with pair gottesfürchtig-säkular: (0.091)


In [217]:
print(holy_unholy_axis.add_pair(("schöpfer", "antichrist")).summary())

Semantic Axis: holy_unholy_de
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.168

Best pairs (highest parallelism):
  ('göttlich', 'satanisch'): 0.228
  ('schöpfer', 'antichrist'): 0.155
  ('cherub', 'teufel'): 0.120

Worst pairs (lowest parallelism):
  ('cherub', 'teufel'): 0.120
  ('schöpfer', 'antichrist'): 0.155
  ('göttlich', 'satanisch'): 0.228


## Holy-Unholy Axis Overview

In [ ]:
holy_unholy_axis = SemAxis(
    [
        ("göttlich", "satanisch"),
        ("schöpfer", "antichrist"),
        ("cherub", "teufel")
    ], 
    w2v_model, 
    name="holy_unholy_de"
)
print(holy_unholy_axis.summary())

Semantic Axis: holy_unholy
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.220

Best pairs (highest parallelism):
  ('divine', 'satanic'): 0.287
  ('angelic', 'demonic'): 0.278
  ('blessed', 'cursed'): 0.184

Worst pairs (lowest parallelism):
  ('sacred', 'profane'): 0.172
  ('holy', 'unholy'): 0.180
  ('blessed', 'cursed'): 0.184


## Spiritual-Material Axis

In [ ]:
# Adding ethereal-mundane
spiritual_material_axis = SemAxis(
    [
        ("kosmisch", "weltlich"),
        ("galaktisch", "irdisch"),
    ], 
    w2v_model, 
    name="spiritual_material_de"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.304

Best pairs (highest parallelism):
  ('kosmisch', 'weltlich'): 0.304
  ('galaktisch', 'irdisch'): 0.304

Worst pairs (lowest parallelism):
  ('kosmisch', 'weltlich'): 0.304
  ('galaktisch', 'irdisch'): 0.304


In [162]:
# Find the 10 closest tokens to the selected token
sim_token = "geistlich"
most_similar_results = w2v_model.most_similar(sim_token, topn=20)
print(f"Tokens closest to '{sim_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'geistlich':
geistig              0.510
sanftmütigem         0.464
korinther            0.462
erettet              0.452
spirituell           0.447
weltliebe            0.446
dreifaltige          0.442
wahlschrei           0.432
seelisch             0.432
gottlos              0.431
1tim                 0.431
materialistisch      0.429
götzen               0.427
katharismus          0.426
christum             0.426
erlösungsplan        0.424
evangelien           0.423
fleischlich          0.422
selig                0.422
gottessohnschaft     0.421


In [184]:
# Looking for top N best antonyms
f, r = spiritual_material_axis.find_antonyms_fullsearch("ätherisch", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: ان - 0.251443217198054
Word-Score: weltlichen - 0.25075360139211017
Word-Score: weltliche - 0.25067319969336194
Word-Score: الل - 0.24919268737236658
Word-Score: passahlamm - 0.2489012579123179
Word-Score: religiöses - 0.24886387089888254
Word-Score: röm - 0.24830090751250586
Word-Score: bekehrt - 0.2482562189300855
Word-Score: mamon - 0.2477425510684649
Word-Score: gleichviel - 0.24770831813414892
Word-Score: seelenheil - 0.2472981077929338
Word-Score: waldenser - 0.2469401334722837
Word-Score: zermonien - 0.24685475354393324
Word-Score: episkopat - 0.24665960421164831
Word-Score: yebamoth - 0.24663280447324118
Word-Score: galiläer - 0.24658303707838058
Word-Score: monotheismus - 0.2462942749261856
Word-Score: knabenschänder - 0.24573374291261038
Word-Score: bibelerzählung - 0.24522887915372849
Word-Score: weltlicher - 0.24501758317152658
Word-Score: gottseligkeit - 0.24500701079765955
Word-Score: juedischen - 0.24471823374430338
Word-Score: liebelehre - 0.2446073244015375

In [ ]:
# Adding ätherisch-fleischlich
spiritual_material_axis = SemAxis(
    [
        ("kosmisch", "weltlich"),
        ("galaktisch", "irdisch"),
        ("ätherisch", "fleischlich"),
    ], 
    w2v_model, 
    name="spiritual_material_de"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.186

Best pairs (highest parallelism):
  ('galaktisch', 'irdisch'): 0.236
  ('kosmisch', 'weltlich'): 0.194
  ('ätherisch', 'fleischlich'): 0.127

Worst pairs (lowest parallelism):
  ('ätherisch', 'fleischlich'): 0.127
  ('kosmisch', 'weltlich'): 0.194
  ('galaktisch', 'irdisch'): 0.236


In [210]:
# Find the 10 closest tokens to the selected token
sim_token = "energie"
most_similar_results = w2v_model.most_similar(sim_token, topn=20)
print(f"Tokens closest to '{sim_token}':")
for tok, sim_score in most_similar_results:
    print(f"{tok:20s} {sim_score:.3f}")

Tokens closest to 'energie':
energien             0.598
murmelgrösse         0.520
teslatechnik         0.518
msed                 0.518
enegie               0.516
austesla             0.511
allfälligi           0.510
engergie             0.507
sonnenenergie        0.503
gezeitenkraftwerke   0.501
heilgeräte           0.500
lebensmittelreplikatoren 0.498
zufliesst            0.497
lebensenergie        0.494
35g4                 0.494
strom                0.493
wirtschsftskrise     0.490
multihop             0.488
versorgungspreisen   0.487
bewegungsenergie     0.486


In [211]:
# Looking for top N best antonyms
f, r = spiritual_material_axis.find_antonyms_fullsearch("energie", top_n=50, refine_pool=300)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: waldenser - 0.20607628673315048
Word-Score: gleichviel - 0.20586156596740088
Word-Score: bekehrt - 0.20477842539548874
Word-Score: seelenheil - 0.2047476495305697
Word-Score: bedeutsamkeit - 0.20329256604115167
Word-Score: religiöses - 0.2025547002752622
Word-Score: passahlamm - 0.20250741889079413
Word-Score: bischöfin - 0.20198111981153488
Word-Score: weltliche - 0.20174798866113028
Word-Score: gottloser - 0.2017017031709353
Word-Score: weltlicher - 0.20135031640529633
Word-Score: mutterkirche - 0.20117743561665216
Word-Score: fleischgewordenen - 0.20111521830161413
Word-Score: unrechtherrschenden - 0.20107796788215637
Word-Score: chrysostomos - 0.20101053391893706
Word-Score: mamon - 0.2009708285331726
Word-Score: ausplündert - 0.20085953176021576
Word-Score: pädokriminell - 0.20071770250797272
Word-Score: zukehrt - 0.20067292948563895
Word-Score: gottesfürchtigen - 0.20064467440048853
Word-Score: fehlleitet - 0.20051970581213632
Word-Score: ablasszahlungen - 0.200468108

In [226]:
# Checking parallelism of extended antonym pairs
candidates = [
    ("licht", "dunkelheit"),
    ("hell", "finster"),
    ("geist", "körper"),
    ("seele", "materie"),
    ("erwachen", "schlafen"),
    ("spirituell", "materiell"),
    ("energie", "materie")
]

for spiritual, mundane in candidates:
    try:
        newax = spiritual_material_axis.add_pair((spiritual, mundane))
        print(f"Parallelism with pair {spiritual}-{mundane}: ({newax.parallelism_score:.3f})")
    except ValueError as err:
        print(err)
   

Parallelism with pair licht-dunkelheit: (0.098)
Parallelism with pair hell-finster: (0.106)
Parallelism with pair geist-körper: (0.044)
Parallelism with pair seele-materie: (0.067)
Parallelism with pair erwachen-schlafen: (0.063)
Parallelism with pair spirituell-materiell: (0.092)
Parallelism with pair energie-materie: (0.117)


In [235]:
print(spiritual_material_axis.add_pair(("energie", "materie")).summary())

# Seems like light-darkness is not a great fit, so I'll replace it with multidimension-unidimensional

Semantic Axis: spiritual_material
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.117

Best pairs (highest parallelism):
  ('galaktisch', 'irdisch'): 0.183
  ('kosmisch', 'weltlich'): 0.134
  ('ätherisch', 'fleischlich'): 0.103

Worst pairs (lowest parallelism):
  ('energie', 'materie'): 0.049
  ('ätherisch', 'fleischlich'): 0.103
  ('kosmisch', 'weltlich'): 0.134


## Spiritual-Material Axis Overview

In [ ]:
# Overview
spiritual_material_axis = SemAxis(
    [
        ("kosmisch", "weltlich"),
        ("galaktisch", "irdisch"),
        ("ätherisch", "fleischlich"),
    ], 
    w2v_model, 
    name="spiritual_material_de"
)
print(spiritual_material_axis.summary())

Semantic Axis: spiritual_material
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.186

Best pairs (highest parallelism):
  ('galaktisch', 'irdisch'): 0.236
  ('kosmisch', 'weltlich'): 0.194
  ('ätherisch', 'fleischlich'): 0.127

Worst pairs (lowest parallelism):
  ('ätherisch', 'fleischlich'): 0.127
  ('kosmisch', 'weltlich'): 0.194
  ('galaktisch', 'irdisch'): 0.236


## Light-Darkness Axis

In [239]:
# Overview
light_darkness_axis = SemAxis(
    [
        ("licht", "dunkelheit"),
    ], 
    w2v_model, 
    name="light_darkness_de"
)
print(light_darkness_axis.summary())

Semantic Axis: light_darkness_de
Number of antonym pairs: 1
Concept vector dimension: 300


## Dictatorship-Democracy Axis

In [97]:
# Find the 20 nearest neighbors to a word in the w2v_model
word = 'verfassungsmässig'
nearest_neighbors = w2v_model.most_similar(word, topn=20)
print(f"20 nearest neighbors to '{word}':")
for word, similarity in nearest_neighbors:
    print(f"{word}: {similarity:.4f}")

20 nearest neighbors to 'verfassungsmässig':
verbriefter: 0.5614
unmenschenrechte: 0.5466
gestaltungsfreiheit: 0.5446
sondermassnahmen: 0.5383
konstituierend: 0.5376
privatrechte: 0.5361
grundrechtliche: 0.5356
blankoschein: 0.5341
freiheitsräume: 0.5278
sicherheitspolizeigesetz: 0.5255
mehrheitsdiktatur: 0.5252
gewährleistetes: 0.5228
verfassungsrechten: 0.5221
rechtsgleichheit: 0.5218
schuldbetreibung: 0.5179
wirtschaftsfreiheit: 0.5166
hausrechte: 0.5141
unverletzliche: 0.5126
verhältnismäßigkeitsgrundsatzes: 0.5123
gewaltenkontrolle: 0.5108


In [118]:
# Construct new tyranny-resistance axis - note that this is almost a 1-to-1 translation from English
political_axis = SemAxis(
    [
    ("diktatur", "demokratie"),
    ("tyrannei", "freiheit"),
    ("unterdrückung", "selbstbestimmung"),
    ("überwachung", "privatsphäre"),
    ("unrechtsstaat", "rechtsstaat")
    ],
    w2v_model, 
    name="democracy_dictatorship_de"
)

print(political_axis.summary())

Semantic Axis: democracy_dictatorship_de
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.216

Best pairs (highest parallelism):
  ('tyrannei', 'freiheit'): 0.257
  ('diktatur', 'demokratie'): 0.239
  ('unterdrückung', 'selbstbestimmung'): 0.220

Worst pairs (lowest parallelism):
  ('unrechtsstaat', 'rechtsstaat'): 0.174
  ('überwachung', 'privatsphäre'): 0.189
  ('unterdrückung', 'selbstbestimmung'): 0.220


In [111]:
# Looking for top N best antonyms
f, r = political_axis.find_antonyms_fullsearch("überwachung", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: grundgesetz - 0.3794367406211024
Word-Score: htsanwalt - 0.3614434865376296
Word-Score: menschenrechte - 0.3532551765259153
Word-Score: gewaltenteilung - 0.3512840912436051
Word-Score: freiheitsrechte - 0.3471698654511024
Word-Score: freiheiten - 0.34607430997696753
Word-Score: grundrechtlich - 0.342418409581129
Word-Score: grundrechten - 0.3387901307277321
Word-Score: rechtsstaatlichkeit - 0.33348163571229417
Word-Score: unveräußerlich - 0.33193649525217717
Word-Score: gmacher_kanal - 0.3306997477230562
Word-Score: verhandelbar - 0.3283646548148996
Word-Score: parteislogan - 0.32826437057619995
Word-Score: rechtssraatlichkeit - 0.3262662138203137
Word-Score: limburgstehtauf - 0.325728773072935
Word-Score: versammlunf - 0.32448155523099165
Word-Score: tsanwalt - 0.3187461208736542
Word-Score: maskenfreier - 0.31741373113061055
Word-Score: demonstrationsrecht - 0.3173341434771386
Word-Score: chtsanwalt - 0.3142558890601332
Word-Score: rechtfertigungsbedürftig - 0.31400179144

## Dictatorship-Democracy Axis Overview

In [ ]:
political_axis = SemAxis(
    [
    ("demokratie", "diktatur"),
    ("freiheit", "tyrannei"),
    ("selbstbestimmung", "unterdrückung"),
    ("privatsphäre", "überwachung"),
    ("rechtsstaat", "unrechtsstaat")
    ],
    w2v_model, 
    name="democracy_dictatorship_de"
)

print(political_axis.summary())

Semantic Axis: democracy_dictatorship_de
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.216

Best pairs (highest parallelism):
  ('tyrannei', 'freiheit'): 0.257
  ('diktatur', 'demokratie'): 0.239
  ('unterdrückung', 'selbstbestimmung'): 0.220

Worst pairs (lowest parallelism):
  ('unrechtsstaat', 'rechtsstaat'): 0.174
  ('überwachung', 'privatsphäre'): 0.189
  ('unterdrückung', 'selbstbestimmung'): 0.220


## Elites-Population Axis

In [ ]:
# Construct new elite-people axis
elite_axis = SemAxis(
    [
        ('volk', 'elite')
    ],
    w2v_model, 
    name="populace_elites_de"
)
print(elite_axis.summary())

Semantic Axis: elite_population_de
Number of antonym pairs: 1
Concept vector dimension: 300


In [122]:
# Looking for top N best antonyms
f, r = elite_axis.find_antonyms_fullsearch("juden", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: eliten - 0.33470650094286497
Word-Score: zibnisten - 0.3240360251064035
Word-Score: bluttrinkende - 0.3228551344488285
Word-Score: untesuchung - 0.31808105647283125
Word-Score: elitären - 0.3140107899536648
Word-Score: hollywood - 0.31048541196707125
Word-Score: kindersexsklaverei - 0.31002337629762816
Word-Score: luzzato - 0.30814383100558107
Word-Score: molochkult - 0.3073676795360927
Word-Score: ädophile - 0.30045942230137795
Word-Score: elitäre - 0.2965151103799851
Word-Score: okkulte - 0.29461027554316604
Word-Score: pädophile - 0.2920873850739041
Word-Score: bluttrinkender - 0.28865728304970834
Word-Score: kinderopferkults - 0.28843696432171245
Word-Score: illuminaten - 0.28785910036394813
Word-Score: satansimus - 0.2856662418297679
Word-Score: kellervorratskammern - 0.2848066190255827
Word-Score: kinderopferkultes - 0.2834744165564802
Word-Score: satanischen - 0.28323325454698395
Word-Score: satanismus - 0.2809021385391521
Word-Score: pädophilen - 0.2807282436523998


In [191]:
elite_axis = elite_axis.add_pair(('elites', 'populace'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 2
Concept vector dimension: 300
Overall parallelism: 0.245

Best pairs (highest parallelism):
  ('regime', 'population'): 0.245
  ('elites', 'populace'): 0.245

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.245
  ('elites', 'populace'): 0.245


In [192]:
# Find the 20 nearest neighbors to a word in the w2v_model
word = 'elites'
nearest_neighbors = w2v_model.most_similar(word, topn=20)
print(f"20 nearest neighbors to '{word}':")
for word, similarity in nearest_neighbors:
    print(f"{word}: {similarity:.4f}")

# Very useful list showing 'dimensions' of elites I could play around;
# glad to see 'deep_state', one of the defined entities, show up here as well

20 nearest neighbors to 'elites':
elite: 0.7399
elitists: 0.5896
politicians: 0.5612
elitist: 0.5572
bankers: 0.5390
billionaires: 0.5304
celebrities: 0.5213
hollywood: 0.5201
cabal: 0.5112
luciferian: 0.5078
technocrats: 0.5002
celebs: 0.4966
sociopaths: 0.4935
psychopaths: 0.4933
satanic: 0.4924
puppets: 0.4898
luciferians: 0.4863
deep_state: 0.4821
illuminati: 0.4813
peasants: 0.4725


In [193]:
# Looking for top N best antonyms
f, r = elite_axis.find_antonyms_fullsearch("billionaires", top_n=50, refine_pool=200)

for word, score in f:
    print(f"Word-Score: {word} - {score}")

Word-Score: populations - 0.42116660351167123
Word-Score: populous - 0.37632938953179057
Word-Score: mwuhahaha - 0.36900968793673683
Word-Score: km² - 0.3606816599684296
Word-Score: 550million - 0.34902208562226106
Word-Score: percentages - 0.3458405023066814
Word-Score: manageable - 0.34531344790751506
Word-Score: uptake - 0.3389820815110177
Word-Score: proportion - 0.33778567438463725
Word-Score: aegypti - 0.33191225920385536
Word-Score: agriculturally - 0.33182480634095723
Word-Score: 204m - 0.33050964596756194
Word-Score: 500mil - 0.3298485336093096
Word-Score: demoralizes - 0.32876383266528264
Word-Score: habituate - 0.3280435839410941
Word-Score: neuromodelation - 0.32744205676439275
Word-Score: factoring - 0.3271784472823376
Word-Score: intermix - 0.3258461923449469
Word-Score: landmass - 0.32545699400525674
Word-Score: unsuspecting - 0.323962018331664
Word-Score: percentage - 0.32061477964130836
Word-Score: mosquitoes - 0.31890178190381974
Word-Score: census - 0.318438876878240

In [194]:
elite_axis = elite_axis.add_pair(('politicians', 'voters'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 3
Concept vector dimension: 300
Overall parallelism: 0.228

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.281
  ('politicians', 'voters'): 0.219
  ('regime', 'population'): 0.183

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.183
  ('politicians', 'voters'): 0.219
  ('elites', 'populace'): 0.281


In [195]:
# Legitimate versus illegitimate political actors
elite_axis = elite_axis.add_pair(('cabal', 'citizens'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.192

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.261
  ('politicians', 'voters'): 0.195
  ('cabal', 'citizens'): 0.156

Worst pairs (lowest parallelism):
  ('regime', 'population'): 0.155
  ('cabal', 'citizens'): 0.156
  ('politicians', 'voters'): 0.195


In [196]:
# Adding an economic dimension
# Removing regime-population because it seems a bit out of place; that frees 'population' up for a better antonym pair
elite_axis = elite_axis.add_pair(('bankers', 'workers')).remove_pair(("regime", "population"))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 4
Concept vector dimension: 300
Overall parallelism: 0.239

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.258
  ('bankers', 'workers'): 0.250
  ('politicians', 'voters'): 0.233

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.216
  ('politicians', 'voters'): 0.233
  ('bankers', 'workers'): 0.250


In [197]:
# More bio-power-inspired
elite_axis = elite_axis.add_pair(('technocrats', 'population'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.247

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.298
  ('technocrats', 'population'): 0.259
  ('politicians', 'voters'): 0.252

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.198
  ('bankers', 'workers'): 0.228
  ('politicians', 'voters'): 0.252


In [198]:
# Adding a celeb-'everyman' axis, given the implication of celebs in Adrenochrome theories
elite_axis = elite_axis.add_pair(('celebrities', 'people'))
print(elite_axis.summary())

Semantic Axis: elite-population
Number of antonym pairs: 6
Concept vector dimension: 300
Overall parallelism: 0.247

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.294
  ('politicians', 'voters'): 0.265
  ('technocrats', 'population'): 0.257

Worst pairs (lowest parallelism):
  ('cabal', 'citizens'): 0.207
  ('bankers', 'workers'): 0.210
  ('celebrities', 'people'): 0.247


In [199]:
# Would like to have something like 'Illuminati' in there but not sure how to pair it
f, r = elite_axis.find_antonyms_fullsearch("satanists", top_n=50, refine_pool=500)

for word, score, _ in r:
    print(f"Word-Score: {word} - {score}")

Word-Score: populous - 0.2560258223896935
Word-Score: residents - 0.2554384803488141
Word-Score: adults - 0.2466498762369156
Word-Score: disenfranchising - 0.24639764286222912
Word-Score: americans - 0.24613488430068606
Word-Score: minorities - 0.24586330425171626
Word-Score: inhabitants - 0.24581257289364225
Word-Score: popasnyansky - 0.24515301202024734
Word-Score: ozernoye - 0.2449595509540467
Word-Score: decennial - 0.24447583513600485
Word-Score: georgians - 0.24444219186192467
Word-Score: gagauzia - 0.24432412286599478
Word-Score: gaotang - 0.24403709386076247
Word-Score: eligible - 0.24388672482399715
Word-Score: canadians - 0.24381743726276217
Word-Score: sapporo - 0.24352494520800455
Word-Score: heinsberg - 0.24350366918813615
Word-Score: workforce - 0.2434515640849159
Word-Score: populations - 0.24326359161308833
Word-Score: californians - 0.24294324857848032
Word-Score: 108k - 0.24294082678499676
Word-Score: zaparozhye - 0.2428469466311591
Word-Score: worker - 0.242767806918

In [ ]:
# Experimented a bit with illuminati and luciferiens; the scores aren't bad, but
# I feel like the terms depart a bit from the original one about elites-vs-masses, so not added
print(elite_axis.add_pair(('luciferians', 'christians')).summary())

Semantic Axis: elite-population
Number of antonym pairs: 7
Concept vector dimension: 300
Overall parallelism: 0.243

Best pairs (highest parallelism):
  ('elites', 'populace'): 0.288
  ('politicians', 'voters'): 0.257
  ('technocrats', 'population'): 0.249

Worst pairs (lowest parallelism):
  ('bankers', 'workers'): 0.204
  ('luciferians', 'christians'): 0.232
  ('celebrities', 'people'): 0.232


## Elites-Population Axis Overview

In [ ]:
elite_axis = SemAxis(
    [
        ('volk', 'elite'),
        ('bürger', 'politiker'),
        ('bevölkerung', 'technokraten'),
        ('menschen', 'promis'),
        ('arbeiter', 'banker')
    ],
    w2v_model, 
    name="populace_elites_de"
)
print(elite_axis.summary())

Semantic Axis: population_elites_de
Number of antonym pairs: 5
Concept vector dimension: 300
Overall parallelism: 0.214

Best pairs (highest parallelism):
  ('menschen', 'promis'): 0.248
  ('bürger', 'politiker'): 0.239
  ('bevölkerung', 'technokraten'): 0.212

Worst pairs (lowest parallelism):
  ('arbeiter', 'banker'): 0.173
  ('volk', 'elite'): 0.200
  ('bevölkerung', 'technokraten'): 0.212


# 3) Comparing Semantic Axes

In [ ]:
# Import German and English model and build matched cross-axis definitions
import json
from pathlib import Path

AXES_EN_PATH = Path("/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/axes_en.json")
AXES_DE_PATH = Path("/project/ssd-stu-research/ploertscher/thesis_code/ideological_resonance_thesis/w2v/analysis/axes_de.json")
MODEL_EN_NAME = "2_w2v_min10"
MODEL_DE_NAME = "2_w2v_min10_de"

# Reuse the English model already loaded above when available; load German here.
en_kv = w2v_model if "w2v_model" in globals() else helpers.load_trained_w2v_keyed_vectors(MODEL_EN_NAME)
de_kv = helpers.load_trained_w2v_keyed_vectors(MODEL_DE_NAME)

token_to_canonical = helpers.load_w2v_token_to_canonical()

with open(AXES_EN_PATH, "r", encoding="utf-8") as f:
    axes_en = {axis: [tuple(pair) for pair in pairs] for axis, pairs in json.load(f).items()}

with open(AXES_DE_PATH, "r", encoding="utf-8") as f:
    axes_de = {axis: [tuple(pair) for pair in pairs] for axis, pairs in json.load(f).items()}


def unit_vector(vec):
    vec = np.asarray(vec, dtype=np.float64)
    norm = np.linalg.norm(vec)
    if norm <= 0:
        raise ValueError("Cannot normalize a zero vector")
    return vec / norm


# Axes match the 2-D plot from semanalysis_plots.py: x=reveal_hide, y=holy_unholy.
reveal_hide_axis_en = unit_vector(SemAxis(axes_en["reveal_hide"], en_kv, name="reveal_hide").concept_vector)
holy_unholy_axis_en = unit_vector(SemAxis(axes_en["holy_unholy"], en_kv, name="holy_unholy").concept_vector)
reveal_hide_axis_de = unit_vector(SemAxis(axes_de["reveal_hide_de"], de_kv, name="reveal_hide_de").concept_vector)
holy_unholy_axis_de = unit_vector(SemAxis(axes_de["holy_unholy_de"], de_kv, name="holy_unholy_de").concept_vector)

language_axes = {
    "EN": {
        "kv": en_kv,
        "x_axis": reveal_hide_axis_en,
        "y_axis": holy_unholy_axis_en,
        "axis_names": ("reveal_hide", "holy_unholy"),
    },
    "DE": {
        "kv": de_kv,
        "x_axis": reveal_hide_axis_de,
        "y_axis": holy_unholy_axis_de,
        "axis_names": ("reveal_hide_de", "holy_unholy_de"),
    },
}

print("Loaded matched EN/DE models and semantic axes for quadrant target-neighbor analysis.")
print(f"Mapped actor tokens available: {len(token_to_canonical)}")

Loaded matched EN/DE models and semantic axes for quadrant target-neighbor analysis.
Mapped actor tokens available: 132


In [ ]:
# Build 300-D target points for each quadrant and inspect their semantic neighborhoods.
# Instead of averaging actors inside a quadrant, anchor each target at the global
# embedding mean and push it to fixed +/-0.5 coordinates on each semantic axis.
TOPN_NEIGHBORS = 50
TARGET_AXIS_OFFSET = 0.5
LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES = {
    "EN": ("_de", "_it"),
    "DE": ("_it", "_us"),
}

quadrants = [
    ("reveal+holy", 1, 1),
    ("reveal+unholy", 1, -1),
    ("hide+holy", -1, 1),
    ("hide+unholy", -1, -1),
]

with open(helpers.ENTITY_MAPPING_2_PATH, "r", encoding="utf-8") as f:
    entity_mapping = json.load(f)

token_to_category = {
    entity_key: category
    for category, entities in entity_mapping.items()
    for entity_key in entities
}


def normalized_vocab_matrix(kv):
    vectors = np.asarray(kv.vectors, dtype=np.float64)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return np.divide(vectors, norms, out=np.zeros_like(vectors), where=norms > 0)


def actor_projection_frame(
    kv,
    x_axis,
    y_axis,
    actor_token_to_canonical,
    token_to_category,
    excluded_category_suffixes=(),
):
    rows = []
    seen_canonicals = set()
    for token, canonical in sorted(actor_token_to_canonical.items(), key=lambda item: item[1]):
        category = token_to_category.get(token)
        if token not in kv or canonical in seen_canonicals:
            continue
        if category is not None and category.endswith(excluded_category_suffixes):
            continue
        vec = unit_vector(kv[token])
        rows.append(
            {
                "token": token,
                "canonical": canonical,
                "category": category,
                "x": float(vec @ x_axis),
                "y": float(vec @ y_axis),
            }
        )
        seen_canonicals.add(canonical)
    return pd.DataFrame(rows)


def quadrant_mask(df, x_sign, y_sign):
    x_ok = df["x"] > 0 if x_sign > 0 else df["x"] < 0
    y_ok = df["y"] > 0 if y_sign > 0 else df["y"] < 0
    return x_ok & y_ok


def neighbor_label(token):
    canonical = token_to_canonical.get(token)
    return f"{token} [{canonical}]" if canonical is not None else token


language_results = {}
summary_rows = []
quadrant_actor_frames = {}

def build_quadrant_targets_and_neighbors(lang, spec, topn=20):
    kv = spec["kv"]
    x_axis = unit_vector(spec["x_axis"])
    y_axis = unit_vector(spec["y_axis"])
    excluded_suffixes = LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES.get(lang, ())
    actor_df = actor_projection_frame(
        kv,
        x_axis,
        y_axis,
        token_to_canonical,
        token_to_category,
        excluded_category_suffixes=excluded_suffixes,
    )
    if actor_df.empty:
        raise ValueError(f"No mapped actors found in {lang} embedding vocabulary.")

    vocab_vectors = normalized_vocab_matrix(kv)
    origin = vocab_vectors.mean(axis=0)

    targets = {
        quadrant_name: origin + x_sign * TARGET_AXIS_OFFSET * x_axis + y_sign * TARGET_AXIS_OFFSET * y_axis
        for quadrant_name, x_sign, y_sign in quadrants
    }
    neighbors = {
        quadrant_name: kv.similar_by_vector(target, topn=topn)
        for quadrant_name, target in targets.items()
    }
    return actor_df, targets, neighbors


for lang, spec in language_axes.items():
    actor_df, targets, neighbors = build_quadrant_targets_and_neighbors(
        lang,
        spec,
        topn=TOPN_NEIGHBORS,
    )
    language_results[lang] = {
        "actor_df": actor_df,
        "targets": targets,
        "neighbors": neighbors,
        "target_offset": TARGET_AXIS_OFFSET,
    }
    quadrant_actor_frames[lang] = actor_df

    excluded_suffixes = LANGUAGE_CATEGORY_EXCLUDE_SUFFIXES.get(lang, ())
    print(f"\n=== {lang}: {spec['axis_names'][0]} x {spec['axis_names'][1]} ===")
    print(f"Actors in embedding vocabulary after language filter: {len(actor_df)}")
    print(f"Excluded actor categories ending with: {excluded_suffixes}")
    print(f"Target quadrant coordinates: +/-{TARGET_AXIS_OFFSET:.2f} on each semantic axis")

    for quadrant_name, x_sign, y_sign in quadrants:
        qdf = actor_df.loc[quadrant_mask(actor_df, x_sign, y_sign)].copy()
        print(f"\n{lang} {quadrant_name}: {len(qdf)} actors in quadrant")
        if not qdf.empty:
            qdf["radius"] = np.sqrt(qdf["x"] ** 2 + qdf["y"] ** 2)
            print("  Most extreme actors in this quadrant's 2-D projection:")
            for row in qdf.sort_values("radius", ascending=False).head(10).itertuples(index=False):
                print(f"    {row.canonical:35s} [{row.category}] x={row.x:+.3f}, y={row.y:+.3f}")

        print(f"  Nearest neighbors to anchored {quadrant_name} target:")
        for rank, (token, score) in enumerate(neighbors[quadrant_name], start=1):
            print(f"    {rank:02d}. {neighbor_label(token):45s} {score:+.3f}")
            summary_rows.append(
                {
                    "language": lang,
                    "quadrant": quadrant_name,
                    "n_actors_in_quadrant": len(qdf),
                    "target_x_coordinate": x_sign * TARGET_AXIS_OFFSET,
                    "target_y_coordinate": y_sign * TARGET_AXIS_OFFSET,
                    "rank": rank,
                    "neighbor": token,
                    "neighbor_label": neighbor_label(token),
                    "similarity": score,
                }
            )

quadrant_target_neighbors = pd.DataFrame(summary_rows)

print("\n=== EN/DE nearest-neighbor comparison by quadrant ===")
for quadrant_name, _x_sign, _y_sign in quadrants:
    print(f"\n── {quadrant_name} ──")
    en_neighbors = language_results["EN"]["neighbors"][quadrant_name]
    de_neighbors = language_results["DE"]["neighbors"][quadrant_name]
    for rank in range(TOPN_NEIGHBORS):
        en_token, en_score = en_neighbors[rank]
        de_token, de_score = de_neighbors[rank]
        print(
            f"{rank + 1:02d}. "
            f"EN {neighbor_label(en_token):38s} {en_score:+.3f}    "
            f"DE {neighbor_label(de_token):38s} {de_score:+.3f}"
        )

display(quadrant_target_neighbors)


=== EN: reveal_hide x holy_unholy ===
Actors in embedding vocabulary after language filter: 60
Excluded actor categories ending with: ('_de', '_it')
Target quadrant coordinates: +/-0.50 on each semantic axis

EN reveal+holy: 16 actors in quadrant
  Most extreme actors in this quadrant's 2-D projection:
    God                                 [spiritual_good] x=+0.120, y=+0.307
    Jesus Christ                        [spiritual_good] x=+0.069, y=+0.286
    Truth Social                        [altmedia_us] x=+0.085, y=+0.110
    Telegram                            [tech] x=+0.086, y=+0.098
    Mike Lindell                        [alt_politics_us] x=+0.058, y=+0.080
    Anons                               [qanon_pro] x=+0.074, y=+0.057
    Patriots                            [qanon_pro] x=+0.039, y=+0.070
    Ghislaine Maxwell                   [globalcabal] x=+0.060, y=+0.029
    Donald Trump                        [donald_trump] x=+0.060, y=+0.029
    Peter McCullough                  

,language,quadrant,n_actors_in_quadrant,target_x_coordinate,target_y_coordinate,rank,neighbor,neighbor_label,similarity
0,EN,reveal+holy,16,0.5,0.5,1,divine,divine,0.495357
1,EN,reveal+holy,16,0.5,0.5,2,gaia,gaia,0.453315
2,EN,reveal+holy,16,0.5,0.5,3,muchlove,muchlove,0.451893
3,EN,reveal+holy,16,0.5,0.5,4,kie,kie,0.448581
4,EN,reveal+holy,16,0.5,0.5,5,blessed,blessed,0.447965
...,...,...,...,...,...,...,...,...,...
395,DE,hide+unholy,15,-0.5,-0.5,46,verfassungsfanaten,verfassungsfanaten,0.394593
396,DE,hide+unholy,15,-0.5,-0.5,47,geschichtsklitterung,geschichtsklitterung,0.394502
397,DE,hide+unholy,15,-0.5,-0.5,48,satanisch,satanisch,0.394439
398,DE,hide+unholy,15,-0.5,-0.5,49,plutoniumbomben,plutoniumbomben,0.393454
